# 02 — Análisis Cualitativo del Dataset LSGoodreads

**Editorial ficticia**: Historia & Biografía
**Fuente de datos**: `src/data/books_clean.csv` (generado por `01_eda_and_cleaning.ipynb`)

Este notebook responde tres preguntas clave para orientar la estrategia editorial:
1. ¿Qué año concentra más publicaciones?
2. ¿Quién es el autor más prolífico del corpus?
3. ¿Qué palabras clave distinguen los libros bien valorados (≥ 4.0) de los peor valorados?

In [ ]:
import os
import ast
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 5)})
sns.set_theme(style="whitegrid", palette="muted")

CSV_PATH = os.path.join("..", "src", "data", "books_clean.csv")

df = pd.read_csv(CSV_PATH)

# genres fue guardado como representación de lista Python; lo convertimos de vuelta
df["genres"] = df["genres"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else []
)

print(f"Libros cargados: {len(df):,}")
df.head(3)

## 1. ¿Qué año concentra más publicaciones?

In [ ]:
year_counts = (
    df.dropna(subset=["year_published"])
      .query("year_published >= 1900 and year_published <= 2026")
      ["year_published"]
      .astype(int)
      .value_counts()
      .sort_index()
)

peak_year  = int(year_counts.idxmax())
peak_count = int(year_counts.max())

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(year_counts.index, year_counts.values, color="steelblue", width=0.8, alpha=0.85)
ax.bar(peak_year, peak_count, color="crimson", width=0.8, label=f"Pico: {peak_year} ({peak_count} libros)")
ax.set(title="Publicaciones por año (desde 1900)", xlabel="Año", ylabel="Número de libros")
ax.xaxis.set_major_locator(mticker.MultipleLocator(5))
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"Año con más publicaciones: {peak_year} ({peak_count} libros)")
print(f"\nTop 10 años más prolíficos:")
print(year_counts.sort_values(ascending=False).head(10).to_string())

### Interpretación

El pico de publicaciones refleja el auge del género de Historia popular en librerías anglosajonas durante esa época. La editorial debe prestar especial atención a los títulos de ese período, ya que constituyen el núcleo del corpus y probablemente el de sus lectores objetivo.

## 2. ¿Quién es el autor más prolífico?

In [ ]:
author_counts = (
    df.groupby("author")
      .size()
      .sort_values(ascending=False)
      .head(25)
)

fig, ax = plt.subplots(figsize=(10, 8))
author_counts.sort_values().plot(kind="barh", ax=ax, color="teal", edgecolor="white", alpha=0.85)
ax.set(title="Top 25 autores más prolíficos en el corpus", xlabel="Número de libros", ylabel="")
plt.tight_layout()
plt.show()

top_author = author_counts.index[0]
top_count  = int(author_counts.iloc[0])
print(f"Autor más prolífico : {top_author} ({top_count} libros)")
print(f"\nTop 10:")
print(author_counts.head(10).to_string())

### Interpretación

Un solo autor dominando el corpus puede indicar tanto un autor muy productivo como un sesgo en las listas scrapeadas. La editorial debe valorar si el catálogo refleja diversidad autorial suficiente o si existe dependencia excesiva en ciertos nombres.

## 3. Análisis de texto: palabras clave asociadas a ratings altos

In [ ]:
# ── 3.1 Separar corpus por rating ────────────────────────────
df_rated = df[df["has_ratings"].eq(True) & df["average_rating"].notna()].copy()

high = df_rated[df_rated["average_rating"] >= 4.0]["book_description"].dropna()
low  = df_rated[df_rated["average_rating"] <  4.0]["book_description"].dropna()

print(f"Libros con rating >= 4.0 : {len(high):,}")
print(f"Libros con rating <  4.0 : {len(low):,}")

In [ ]:
# ── 3.2 TF-IDF sobre todo el corpus ─────────────────────────
STOP = "english"
vec = TfidfVectorizer(
    max_features=500,
    stop_words=STOP,
    ngram_range=(1, 2),
    min_df=5
)
vec.fit(df_rated["book_description"].dropna())

def mean_tfidf(corpus, vectorizer):
    mat = vectorizer.transform(corpus)
    return pd.Series(
        mat.mean(axis=0).A1,
        index=vectorizer.get_feature_names_out()
    ).sort_values(ascending=False)

high_scores = mean_tfidf(high, vec)
low_scores  = mean_tfidf(low,  vec)

# Diferencia: qué términos son más característicos de libros bien valorados
diff = (high_scores - low_scores).sort_values(ascending=False)
print("Top 15 términos más asociados a rating ALTO:")
print(diff.head(15).to_string())

In [ ]:
# ── 3.3 WordCloud — libros con rating alto ──────────────────
wc_high = WordCloud(
    width=900, height=400,
    background_color="white",
    colormap="Blues",
    max_words=100
).generate_from_frequencies(high_scores.to_dict())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].imshow(wc_high, interpolation="bilinear")
axes[0].axis("off")
axes[0].set_title("WordCloud — libros con rating ≥ 4.0", fontsize=13)

wc_low = WordCloud(
    width=900, height=400,
    background_color="white",
    colormap="Reds",
    max_words=100
).generate_from_frequencies(low_scores.to_dict())

axes[1].imshow(wc_low, interpolation="bilinear")
axes[1].axis("off")
axes[1].set_title("WordCloud — libros con rating < 4.0", fontsize=13)

plt.tight_layout()
plt.show()

In [ ]:
# ── 3.4 Barras comparativas top-30 términos diferenciadores ─
top_n = 30
top_diff = diff.head(top_n)

fig, ax = plt.subplots(figsize=(12, 7))
colors = ["seagreen" if v > 0 else "tomato" for v in top_diff.values]
ax.barh(top_diff.index[::-1], top_diff.values[::-1], color=colors[::-1], edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set(title="Términos más diferenciadores entre rating alto vs bajo (TF-IDF medio)",
       xlabel="Diferencia TF-IDF (alto − bajo)")
plt.tight_layout()
plt.show()

### Interpretación

Los términos que aparecen más en libros bien valorados (verde) representan el vocabulario editorial ganador para Historia & Biografía. La editorial debería:
- Priorizar manuscritos que giren en torno a esos conceptos clave.
- Usar esas palabras como señales en el catálogo y en el marketing de títulos.
- Tratar con cautela los términos del grupo rojo (aparecen más en libros peor valorados).